In [1]:
pip install duckdb pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import duckdb

In [3]:
folder = Path("raw_data")

In [4]:
con = duckdb.connect()

In [5]:
for name in ["account_managers", "services", "customers", "aws_usage", "targets"]:
    con.sql(f"create view {name} as select * from read_csv_auto('{folder / (name + '.csv')}')")

In [6]:
print(con.sql("select * from account_managers limit 5"))

┌────────────┬──────────────┐
│ manager_id │ manager_name │
│  varchar   │   varchar    │
├────────────┼──────────────┤
│ AM001      │ James        │
│ AM002      │ Michael      │
│ AM003      │ David        │
│ AM004      │ Sarah        │
│ AM005      │ Emily        │
└────────────┴──────────────┘



In [7]:
result = con.sql("""
    select * from account_managers limit 5
""").df()

In [8]:
print(result)

  manager_id manager_name
0      AM001        James
1      AM002      Michael
2      AM003        David
3      AM004        Sarah
4      AM005        Emily


In [9]:
result.to_csv(folder / "result.csv", index=False)

In [10]:
exercise_1 = con.sql("""
with country_target as (
  select
    month,
    region,
    country,
    company_count as company_count_target
  from targets
  where
    target_id = 1
    and month = date('2026-08-01')
  ),

customer_usage as (
  select
    customer_id,
    sum(service_usage) as service_usage
  from aws_usage
  where
    month = date('2026-08-01')
  group by
    customer_id
),

country_actual as (
  select
    t2.customer_country,
    sum(case when t1.service_usage >= 100 then 1 else 0 end) as company_count_actual
  from customer_usage t1
  left outer join customers t2
    on t1.customer_id = t2.customer_id
  group by
    t2.customer_country
)

select
  t1.month,
  t1.region,
  t1.country,
  t1.company_count_target,
  coalesce(t2.company_count_actual, 0) as company_count_actual,
  case when coalesce(t2.company_count_actual, 0) >= t1.company_count_target then 'Ahead' else 'Missed' end as status
from country_target t1
left outer join country_actual t2
  on t1.country = t2.customer_country
order by
  t1.month,
  t1.region,
  t1.country
""").df()

In [11]:
print(exercise_1)

       month    region    country  company_count_target  company_count_actual  \
0 2026-08-01  Region A  Country 1                   135                 119.0   
1 2026-08-01  Region B  Country 2                   116                 130.0   
2 2026-08-01  Region C  Country 3                   104                 116.0   
3 2026-08-01  Region D  Country 4                    59                  66.0   
4 2026-08-01  Region D  Country 5                    51                  45.0   
5 2026-08-01  Region E  Country 6                    47                  41.0   
6 2026-08-01  Region E  Country 7                    22                  25.0   
7 2026-08-01  Region E  Country 8                    30                  26.0   
8 2026-08-01  Region E  Country 9                    29                  32.0   

   status  
0  Missed  
1   Ahead  
2   Ahead  
3   Ahead  
4  Missed  
5  Missed  
6   Ahead  
7  Missed  
8   Ahead  


In [12]:
exercise_1.to_csv(folder / "exercise_1.csv", index=False)

In [13]:
duplicate_check = con.sql("""
select
  customer_id,
  month,
  service_id,
  count(*) as row_count
from aws_usage
group by
  customer_id,
  month,
  service_id
having count(*) > 1
""").df()

In [14]:
print(duplicate_check)

Empty DataFrame
Columns: [customer_id, month, service_id, row_count]
Index: []


In [15]:
print(con.sql("select distinct service_category from services"))

┌───────────────────────────────┐
│       service_category        │
│            varchar            │
├───────────────────────────────┤
│ Analytics                     │
│ Networking & Content Delivery │
│ Management & Monitoring       │
│ End User Computing            │
│ Compute                       │
│ Storage                       │
│ Database                      │
│ Application Integration       │
│ Machine Learning              │
└───────────────────────────────┘



In [16]:
print(con.sql("select * from services where service_category = 'Analytics'"))

┌────────────┬────────────────────┬──────────────────┐
│ service_id │    service_name    │ service_category │
│  varchar   │      varchar       │     varchar      │
├────────────┼────────────────────┼──────────────────┤
│ SVC010     │ Redshift           │ Analytics        │
│ SVC011     │ Glue               │ Analytics        │
│ SVC012     │ Athena             │ Analytics        │
│ SVC013     │ QuickSight         │ Analytics        │
│ SVC014     │ Kinesis            │ Analytics        │
│ SVC015     │ OpenSearch Service │ Analytics        │
└────────────┴────────────────────┴──────────────────┘



In [17]:
print(con.sql("select count(*) as zero_rows from aws_usage where service_usage <= 0"))

┌───────────┐
│ zero_rows │
│   int64   │
├───────────┤
│         0 │
└───────────┘



In [18]:
analytics_rows = con.sql("""
select
    t1.customer_id,
    t1.service_id,
    t2.service_category
from aws_usage t1
inner join services t2
    on t1.service_id = t2.service_id
where t1.month = date('2026-08-01')
  and t2.service_category = 'Analytics'
""").df()

print(analytics_rows.head(10))
print("行数：", len(analytics_rows))

  customer_id service_id service_category
0   CUST00002     SVC011        Analytics
1   CUST00002     SVC012        Analytics
2   CUST00002     SVC013        Analytics
3   CUST00002     SVC010        Analytics
4   CUST00003     SVC011        Analytics
5   CUST00007     SVC012        Analytics
6   CUST00007     SVC013        Analytics
7   CUST00007     SVC010        Analytics
8   CUST00012     SVC011        Analytics
9   CUST00012     SVC013        Analytics
行数： 499


In [19]:
analytics_customers = con.sql("""
select distinct
    t1.customer_id
from aws_usage t1
inner join services t2
    on t1.service_id = t2.service_id
where t1.month = date('2026-08-01')
  and t2.service_category = 'Analytics'
""").df()

print("不重复客户总数：", len(analytics_customers))

不重复客户总数： 255


In [20]:
assignment = con.sql("""
with analytics_customers as (
    select distinct
        t1.customer_id
    from aws_usage t1
    inner join services t2
        on t1.service_id = t2.service_id
    where t1.month = date('2026-08-01')
      and t2.service_category = 'Analytics'
)
select
    t2.customer_country as country,
    count(*) as unique_customers
from analytics_customers t1
left outer join customers t2
    on t1.customer_id = t2.customer_id
group by t2.customer_country
order by t2.customer_country
""").df()

print(assignment)

     country  unique_customers
0  Country 1                66
1  Country 2                47
2  Country 3                44
3  Country 4                31
4  Country 5                16
5  Country 6                16
6  Country 7                 9
7  Country 8                12
8  Country 9                14


In [21]:
print("各国家相加：", assignment["unique_customers"].sum())
print("不重复客户总数：", len(analytics_customers))

各国家相加： 255
不重复客户总数： 255


In [22]:
assignment_v2 = con.sql("""
select
    t3.customer_country as country,
    count(distinct t1.customer_id) as unique_customers
from aws_usage t1
inner join services t2
    on t1.service_id = t2.service_id
left outer join customers t3
    on t1.customer_id = t3.customer_id
where t1.month = date('2026-08-01')
  and t2.service_category = 'Analytics'
group by t3.customer_country
order by t3.customer_country
""").df()

check = assignment.merge(assignment_v2, on="country", suffixes=("_v1", "_v2"))
check["same"] = check["unique_customers_v1"] == check["unique_customers_v2"]
print(check)
print("两种方法完全一致：", check["same"].all())

     country  unique_customers_v1  unique_customers_v2  same
0  Country 1                   66                   66  True
1  Country 2                   47                   47  True
2  Country 3                   44                   44  True
3  Country 4                   31                   31  True
4  Country 5                   16                   16  True
5  Country 6                   16                   16  True
6  Country 7                    9                    9  True
7  Country 8                   12                   12  True
8  Country 9                   14                   14  True
两种方法完全一致： True


In [23]:
assignment_full = con.sql("""
with analytics_customers as (
    select distinct t1.customer_id
    from aws_usage t1
    inner join services t2 on t1.service_id = t2.service_id
    where t1.month = date('2026-08-01')
      and t2.service_category = 'Analytics'
),
country_actual as (
    select t2.customer_country, count(*) as unique_customers
    from analytics_customers t1
    left outer join customers t2 on t1.customer_id = t2.customer_id
    group by t2.customer_country
),
all_countries as (
    select distinct customer_country from customers
)
select
    t1.customer_country as country,
    coalesce(t2.unique_customers, 0) as unique_customers
from all_countries t1
left outer join country_actual t2
    on t1.customer_country = t2.customer_country
order by t1.customer_country
""").df()

print(assignment_full)

     country  unique_customers
0  Country 1                66
1  Country 2                47
2  Country 3                44
3  Country 4                31
4  Country 5                16
5  Country 6                16
6  Country 7                 9
7  Country 8                12
8  Country 9                14


In [24]:
assignment.to_csv(folder / "assignment.csv", index=False)
print("已保存到：", folder / "assignment.csv")

已保存到： raw_data/assignment.csv


In [25]:
assignment_target = con.sql("""
with country_target as (
    select month, region, country, company_count as target
    from targets
    where target_id = 2
      and month = date('2026-08-01')
),
analytics_customers as (
    select distinct t1.customer_id
    from aws_usage t1
    inner join services t2 on t1.service_id = t2.service_id
    where t1.month = date('2026-08-01')
      and t2.service_category = 'Analytics'
),
country_actual as (
    select t2.customer_country, count(*) as unique_customers
    from analytics_customers t1
    left outer join customers t2 on t1.customer_id = t2.customer_id
    group by t2.customer_country
)
select
    t1.month,
    t1.region,
    t1.country,
    t1.target,
    coalesce(t2.unique_customers, 0) as unique_customers,
    case when coalesce(t2.unique_customers, 0) >= t1.target
         then 'Ahead' else 'Missed' end as status
from country_target t1
left outer join country_actual t2
    on t1.country = t2.customer_country
order by t1.month, t1.region, t1.country
""").df()

print(assignment_target)
assignment_target.to_csv(folder / "assignment_target.csv", index=False)

       month    region    country  target  unique_customers  status
0 2026-08-01  Region A  Country 1      75                66  Missed
1 2026-08-01  Region B  Country 2      42                47   Ahead
2 2026-08-01  Region C  Country 3      39                44   Ahead
3 2026-08-01  Region D  Country 4      28                31   Ahead
4 2026-08-01  Region D  Country 5      18                16  Missed
5 2026-08-01  Region E  Country 6      18                16  Missed
6 2026-08-01  Region E  Country 7       8                 9   Ahead
7 2026-08-01  Region E  Country 8      14                12  Missed
8 2026-08-01  Region E  Country 9      13                14   Ahead


In [26]:
low_hanging_fruit = con.sql("""
with customer_usage as (
    select
        customer_id,
        sum(service_usage) as service_usage
    from aws_usage
    where month = date('2026-08-01')
    group by customer_id
)
select
    t1.customer_id,
    t2.customer_name,
    t2.customer_country as country,
    t2.customer_region as region,
    round(t1.service_usage, 2) as service_usage,
    t3.manager_name
from customer_usage t1
left outer join customers t2
    on t1.customer_id = t2.customer_id
left outer join account_managers t3
    on t2.manager_id = t3.manager_id
where t1.service_usage >= 80
  and t1.service_usage < 100
order by t1.service_usage desc
""").df()

print(low_hanging_fruit.head(10))
print("行数：", len(low_hanging_fruit))

low_hanging_fruit.to_csv(folder / "low_hanging_fruit.csv", index=False)

  customer_id     customer_name    country    region  service_usage  \
0   CUST00853     Mada Holdings  Country 6  Region E          99.96   
1   CUST00324   Krogat Holdings  Country 1  Region A          99.72   
2   CUST00849      Jaigas Foods  Country 4  Region D          99.67   
3   CUST00169   Frular Partners  Country 2  Region B          99.45   
4   CUST00030     Klijiot Foods  Country 4  Region D          99.45   
5   CUST00234  Dutovai Dynamics  Country 1  Region A          99.34   
6   CUST00449    Digor Partners  Country 2  Region B          99.31   
7   CUST00340        Fege Group  Country 1  Region A          99.20   
8   CUST00889  Maipais Partners  Country 6  Region E          98.49   
9   CUST00460   Gujir Analytics  Country 5  Region D          97.88   

  manager_name  
0     Kimberly  
1        Sarah  
2       Thomas  
3     Michelle  
4  Christopher  
5        James  
6      Matthew  
7      Michael  
8       Steven  
9        Kevin  
行数： 57
